In [1]:
print("Hi")

Hi


In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Load raw datasets
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")

print("Raw data loaded successfully")

Raw data loaded successfully


In [3]:
# Remove exact duplicate rows from all source tables
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
customers = customers.drop_duplicates()
products = products.drop_duplicates()
payments = payments.drop_duplicates()
reviews = reviews.drop_duplicates()
sellers = sellers.drop_duplicates()
category_translation = category_translation.drop_duplicates()

# Convert order date columns
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Convert shipping limit date
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"], errors="coerce"
)

# Convert review dates
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"], errors="coerce"
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"], errors="coerce"
)

# Convert numeric columns
order_items["price"] = pd.to_numeric(order_items["price"], errors="coerce")
order_items["freight_value"] = pd.to_numeric(order_items["freight_value"], errors="coerce")
payments["payment_value"] = pd.to_numeric(payments["payment_value"], errors="coerce")
payments["payment_installments"] = pd.to_numeric(payments["payment_installments"], errors="coerce")

print("Basic cleaning and data type conversion completed")

Basic cleaning and data type conversion completed


In [4]:
dim_customer = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].copy()

# Remove duplicate customer_id
dim_customer = dim_customer.drop_duplicates(subset=["customer_id"])

# Standardize text columns
dim_customer["customer_city"] = dim_customer["customer_city"].str.strip().str.title()
dim_customer["customer_state"] = dim_customer["customer_state"].str.strip().str.upper()

# Create surrogate key
dim_customer = dim_customer.reset_index(drop=True)
dim_customer.insert(0, "customer_key", dim_customer.index + 1)

print("Dim_Customer created")
print(dim_customer.shape)
dim_customer.head()

Dim_Customer created
(99441, 6)


,customer_key,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
1,2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,Sao Bernardo Do Campo,SP
2,3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,Sao Paulo,SP
3,4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,Mogi Das Cruzes,SP
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP


In [5]:
# Join products with category translation
dim_product = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

# Select required columns
dim_product = dim_product[
    [
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].copy()

# Remove duplicate product_id
dim_product = dim_product.drop_duplicates(subset=["product_id"])

# Handle missing category
dim_product["product_category_name"] = dim_product["product_category_name"].fillna("unknown")
dim_product["product_category_name_english"] = dim_product["product_category_name_english"].fillna("Unknown")

# Convert numeric product columns
product_numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in product_numeric_cols:
    dim_product[col] = pd.to_numeric(dim_product[col], errors="coerce")

# Create product volume
dim_product["product_volume_cm3"] = (
    dim_product["product_length_cm"] *
    dim_product["product_height_cm"] *
    dim_product["product_width_cm"]
)

# Create surrogate key
dim_product = dim_product.reset_index(drop=True)
dim_product.insert(0, "product_key", dim_product.index + 1)

print("Dim_Product created")
print(dim_product.shape)
dim_product.head()

Dim_Product created
(32951, 12)


,product_key,product_id,product_category_name,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3
0,1,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0,2240.0
1,2,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,10800.0
2,3,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0,2430.0
3,4,cef67bcfe19066a932b7673e239eb23d,bebes,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0,2704.0
4,5,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0,4420.0


In [6]:
dim_seller = sellers[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].copy()

# Remove duplicate sellers
dim_seller = dim_seller.drop_duplicates(subset=["seller_id"])

# Standardize text columns
dim_seller["seller_city"] = dim_seller["seller_city"].str.strip().str.title()
dim_seller["seller_state"] = dim_seller["seller_state"].str.strip().str.upper()

# Create surrogate key
dim_seller = dim_seller.reset_index(drop=True)
dim_seller.insert(0, "seller_key", dim_seller.index + 1)

print("Dim_Seller created")
print(dim_seller.shape)
dim_seller.head()

Dim_Seller created
(3095, 5)


,seller_key,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,1,3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP
1,2,d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi Guacu,SP
2,3,ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio De Janeiro,RJ
3,4,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,Sao Paulo,SP
4,5,51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca Paulista,SP


In [7]:
# Collect date columns from orders
date_series_list = [
    orders["order_purchase_timestamp"],
    orders["order_approved_at"],
    orders["order_delivered_carrier_date"],
    orders["order_delivered_customer_date"],
    orders["order_estimated_delivery_date"],
    order_items["shipping_limit_date"]
]

# Combine all dates
all_dates = pd.concat(date_series_list)

# Convert to only date part
all_dates = pd.to_datetime(all_dates, errors="coerce").dt.date

# Remove nulls and duplicates
unique_dates = pd.Series(all_dates.dropna().unique())

# Create Dim_Date
dim_date = pd.DataFrame({
    "full_date": pd.to_datetime(unique_dates)
})

# Sort dates
dim_date = dim_date.sort_values("full_date").reset_index(drop=True)

# Create date_key in YYYYMMDD format
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)

# Create date attributes
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter"] = "Q" + dim_date["full_date"].dt.quarter.astype(str)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["week_of_year"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"] = np.where(dim_date["day_name"].isin(["Saturday", "Sunday"]), 1, 0)

# Reorder columns
dim_date = dim_date[
    [
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "week_of_year",
        "day_name",
        "is_weekend"
    ]
]

print("Dim_Date created")
print(dim_date.shape)
dim_date.head()

Dim_Date created
(720, 10)


,date_key,full_date,day,month,month_name,quarter,year,week_of_year,day_name,is_weekend
0,20160904,2016-09-04,4,9,September,Q3,2016,35,Sunday,1
1,20160905,2016-09-05,5,9,September,Q3,2016,36,Monday,0
2,20160913,2016-09-13,13,9,September,Q3,2016,37,Tuesday,0
3,20160915,2016-09-15,15,9,September,Q3,2016,37,Thursday,0
4,20160919,2016-09-19,19,9,September,Q3,2016,38,Monday,0


In [8]:
payment_agg = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_methods_count=("payment_type", "nunique"),
    max_payment_installments=("payment_installments", "max")
).reset_index()

# Primary payment type = payment type with highest payment value per order
primary_payment = payments.sort_values(
    ["order_id", "payment_value"],
    ascending=[True, False]
).drop_duplicates(subset=["order_id"])[["order_id", "payment_type"]]

primary_payment = primary_payment.rename(columns={"payment_type": "primary_payment_type"})

payment_agg = payment_agg.merge(primary_payment, on="order_id", how="left")

print("Payment aggregation created")
print(payment_agg.shape)
payment_agg.head()

Payment aggregation created
(99440, 5)


,order_id,total_payment_value,payment_methods_count,max_payment_installments,primary_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,credit_card


In [9]:
review_agg = reviews.groupby("order_id").agg(
    review_score=("review_score", "mean"),
    review_creation_date=("review_creation_date", "max"),
    review_answer_timestamp=("review_answer_timestamp", "max")
).reset_index()

# Round review score
review_agg["review_score"] = review_agg["review_score"].round(2)

print("Review aggregation created")
print(review_agg.shape)
review_agg.head()

Review aggregation created
(98673, 4)


,order_id,review_score,review_creation_date,review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,2017-09-21,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,4.0,2017-05-13,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,5.0,2018-01-23,2018-01-23 16:06:31
3,00024acbcdf0a6daa1e931b038114c75,4.0,2018-08-15,2018-08-15 16:39:01
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,2017-03-02,2017-03-03 10:54:59


In [10]:
order_items_agg = order_items.groupby("order_id").agg(
    total_items=("order_item_id", "count"),
    total_product_value=("price", "sum"),
    total_freight_value=("freight_value", "sum")
).reset_index()

order_items_agg["total_order_value"] = (
    order_items_agg["total_product_value"] +
    order_items_agg["total_freight_value"]
)

print("Order items aggregation created")
print(order_items_agg.shape)
order_items_agg.head()

Order items aggregation created
(98666, 5)


,order_id,total_items,total_product_value,total_freight_value,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [11]:
customer_lookup = dim_customer[["customer_key", "customer_id"]]
product_lookup = dim_product[["product_key", "product_id"]]
seller_lookup = dim_seller[["seller_key", "seller_id"]]
date_lookup = dim_date[["date_key", "full_date"]].copy()

# Convert full_date to date for joining
date_lookup["full_date"] = pd.to_datetime(date_lookup["full_date"]).dt.date

print("Lookup tables created")

Lookup tables created


In [12]:
# Start from order_items
fact_order_items = order_items.copy()

# Join with orders to get customer and purchase timestamp
fact_order_items = fact_order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp"
        ]
    ],
    on="order_id",
    how="left"
)

# Add customer_key
fact_order_items = fact_order_items.merge(
    customer_lookup,
    on="customer_id",
    how="left"
)

# Add product_key
fact_order_items = fact_order_items.merge(
    product_lookup,
    on="product_id",
    how="left"
)

# Add seller_key
fact_order_items = fact_order_items.merge(
    seller_lookup,
    on="seller_id",
    how="left"
)

# Create date fields for lookup
fact_order_items["purchase_full_date"] = pd.to_datetime(
    fact_order_items["order_purchase_timestamp"], errors="coerce"
).dt.date

fact_order_items["shipping_limit_full_date"] = pd.to_datetime(
    fact_order_items["shipping_limit_date"], errors="coerce"
).dt.date

# Add purchase_date_key
fact_order_items = fact_order_items.merge(
    date_lookup.rename(columns={
        "full_date": "purchase_full_date",
        "date_key": "purchase_date_key"
    }),
    on="purchase_full_date",
    how="left"
)

# Add shipping_limit_date_key
fact_order_items = fact_order_items.merge(
    date_lookup.rename(columns={
        "full_date": "shipping_limit_full_date",
        "date_key": "shipping_limit_date_key"
    }),
    on="shipping_limit_full_date",
    how="left"
)

# Derived columns
fact_order_items["item_total_value"] = (
    fact_order_items["price"] + fact_order_items["freight_value"]
)

fact_order_items["item_count"] = 1

# Audit columns
fact_order_items["load_batch_id"] = "FULL_LOAD_001"
fact_order_items["record_inserted_at"] = datetime.now()

# Select final columns
fact_order_items = fact_order_items[
    [
        "order_id",
        "order_item_id",
        "customer_key",
        "product_key",
        "seller_key",
        "purchase_date_key",
        "shipping_limit_date_key",
        "order_status",
        "price",
        "freight_value",
        "item_total_value",
        "item_count",
        "order_purchase_timestamp",
        "shipping_limit_date",
        "load_batch_id",
        "record_inserted_at"
    ]
]

print("Fact_Order_Items created")
print(fact_order_items.shape)
fact_order_items.head()

Fact_Order_Items created
(112650, 16)


,order_id,order_item_id,customer_key,product_key,seller_key,purchase_date_key,shipping_limit_date_key,order_status,price,freight_value,item_total_value,item_count,order_purchase_timestamp,shipping_limit_date,load_batch_id,record_inserted_at
0,00010242fe8c5a6d1ba2dd792cb16214,1,65558,25866,514,20170913,20170919,delivered,58.90,13.29,72.19,1,2017-09-13 08:59:02,2017-09-19 09:45:35,FULL_LOAD_001,2026-05-23 17:52:26.680184
1,00018f77f2f0320c557190d7a144bdd3,1,34266,27231,472,20170426,20170503,delivered,239.90,19.93,259.83,1,2017-04-26 10:53:06,2017-05-03 11:05:13,FULL_LOAD_001,2026-05-23 17:52:26.680184
2,000229ec398224ef6ca0657da4fc703e,1,34956,22625,1825,20180114,20180118,delivered,199.00,17.87,216.87,1,2018-01-14 14:33:31,2018-01-18 14:48:30,FULL_LOAD_001,2026-05-23 17:52:26.680184
3,00024acbcdf0a6daa1e931b038114c75,1,51764,15404,2024,20180808,20180815,delivered,12.99,12.79,25.78,1,2018-08-08 10:00:35,2018-08-15 10:10:18,FULL_LOAD_001,2026-05-23 17:52:26.680184
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,7603,8863,1598,20170204,20170213,delivered,199.90,18.14,218.04,1,2017-02-04 13:57:51,2017-02-13 13:57:51,FULL_LOAD_001,2026-05-23 17:52:26.680184


In [13]:
# Start from orders
fact_order_summary = orders.copy()

# Add customer_key
fact_order_summary = fact_order_summary.merge(
    customer_lookup,
    on="customer_id",
    how="left"
)

# Add item aggregation
fact_order_summary = fact_order_summary.merge(
    order_items_agg,
    on="order_id",
    how="left"
)

# Add payment aggregation
fact_order_summary = fact_order_summary.merge(
    payment_agg,
    on="order_id",
    how="left"
)

# Add review aggregation
fact_order_summary = fact_order_summary.merge(
    review_agg,
    on="order_id",
    how="left"
)

# Create date fields
fact_order_summary["purchase_full_date"] = pd.to_datetime(
    fact_order_summary["order_purchase_timestamp"], errors="coerce"
).dt.date

fact_order_summary["approved_full_date"] = pd.to_datetime(
    fact_order_summary["order_approved_at"], errors="coerce"
).dt.date

fact_order_summary["delivered_customer_full_date"] = pd.to_datetime(
    fact_order_summary["order_delivered_customer_date"], errors="coerce"
).dt.date

fact_order_summary["estimated_delivery_full_date"] = pd.to_datetime(
    fact_order_summary["order_estimated_delivery_date"], errors="coerce"
).dt.date

# Add purchase_date_key
fact_order_summary = fact_order_summary.merge(
    date_lookup.rename(columns={
        "full_date": "purchase_full_date",
        "date_key": "purchase_date_key"
    }),
    on="purchase_full_date",
    how="left"
)

# Add approved_date_key
fact_order_summary = fact_order_summary.merge(
    date_lookup.rename(columns={
        "full_date": "approved_full_date",
        "date_key": "approved_date_key"
    }),
    on="approved_full_date",
    how="left"
)

# Add delivered_customer_date_key
fact_order_summary = fact_order_summary.merge(
    date_lookup.rename(columns={
        "full_date": "delivered_customer_full_date",
        "date_key": "delivered_customer_date_key"
    }),
    on="delivered_customer_full_date",
    how="left"
)

# Add estimated_delivery_date_key
fact_order_summary = fact_order_summary.merge(
    date_lookup.rename(columns={
        "full_date": "estimated_delivery_full_date",
        "date_key": "estimated_delivery_date_key"
    }),
    on="estimated_delivery_full_date",
    how="left"
)

# Fill missing numeric values
numeric_fill_cols = [
    "total_items",
    "total_product_value",
    "total_freight_value",
    "total_order_value",
    "total_payment_value",
    "payment_methods_count",
    "max_payment_installments"
]

for col in numeric_fill_cols:
    fact_order_summary[col] = fact_order_summary[col].fillna(0)

# Fill missing primary payment type
fact_order_summary["primary_payment_type"] = fact_order_summary["primary_payment_type"].fillna("Unknown")

# Delivery metrics
fact_order_summary["delivery_days"] = (
    fact_order_summary["order_delivered_customer_date"] -
    fact_order_summary["order_purchase_timestamp"]
).dt.days

fact_order_summary["estimated_delivery_days"] = (
    fact_order_summary["order_estimated_delivery_date"] -
    fact_order_summary["order_purchase_timestamp"]
).dt.days

fact_order_summary["delivery_delay_days"] = (
    fact_order_summary["order_delivered_customer_date"] -
    fact_order_summary["order_estimated_delivery_date"]
).dt.days

# Flags
fact_order_summary["is_delivered"] = np.where(
    fact_order_summary["order_status"] == "delivered", 1, 0
)

fact_order_summary["is_late_delivery"] = np.where(
    fact_order_summary["delivery_delay_days"] > 0, 1, 0
)

fact_order_summary["is_cancelled"] = np.where(
    fact_order_summary["order_status"] == "canceled", 1, 0
)

# Audit columns
fact_order_summary["load_batch_id"] = "FULL_LOAD_001"
fact_order_summary["record_inserted_at"] = datetime.now()

# Select final columns
fact_order_summary = fact_order_summary[
    [
        "order_id",
        "customer_key",
        "purchase_date_key",
        "approved_date_key",
        "delivered_customer_date_key",
        "estimated_delivery_date_key",
        "order_status",
        "total_items",
        "total_product_value",
        "total_freight_value",
        "total_order_value",
        "total_payment_value",
        "payment_methods_count",
        "max_payment_installments",
        "primary_payment_type",
        "review_score",
        "delivery_days",
        "estimated_delivery_days",
        "delivery_delay_days",
        "is_delivered",
        "is_late_delivery",
        "is_cancelled",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "load_batch_id",
        "record_inserted_at"
    ]
]

print("Fact_Order_Summary created")
print(fact_order_summary.shape)
fact_order_summary.head()

Fact_Order_Summary created
(99441, 28)


,order_id,customer_key,purchase_date_key,approved_date_key,delivered_customer_date_key,estimated_delivery_date_key,order_status,total_items,total_product_value,total_freight_value,...,delivery_delay_days,is_delivered,is_late_delivery,is_cancelled,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date,load_batch_id,record_inserted_at
0,e481f51cbdc54678b7cc49136f2d6af7,70297,20171002,20171002.0,20171010.0,20171018,delivered,1.0,29.99,8.72,...,-8.0,1,0,0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13,2017-10-18,FULL_LOAD_001,2026-05-23 17:53:42.672131
1,53cdb2fc8bc7dce0b6741e2150273451,77028,20180724,20180726.0,20180807.0,20180813,delivered,1.0,118.70,22.76,...,-6.0,1,0,0,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45,2018-08-13,FULL_LOAD_001,2026-05-23 17:53:42.672131
2,47770eb9100c2d0c44946d9cf07ec65d,555,20180808,20180808.0,20180817.0,20180904,delivered,1.0,159.90,19.22,...,-18.0,1,0,0,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29,2018-09-04,FULL_LOAD_001,2026-05-23 17:53:42.672131
3,949d5b44dbf5de918fe9c16f97b45f8a,61082,20171118,20171118.0,20171202.0,20171215,delivered,1.0,45.00,27.20,...,-13.0,1,0,0,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42,2017-12-15,FULL_LOAD_001,2026-05-23 17:53:42.672131
4,ad21c59c0840e6cb83a9ceb5573f8159,67264,20180213,20180213.0,20180216.0,20180226,delivered,1.0,19.90,8.72,...,-10.0,1,0,0,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02,2018-02-26,FULL_LOAD_001,2026-05-23 17:53:42.672131


In [14]:
print("===== DIMENSION KEY VALIDATION =====")

print("Dim_Customer customer_key unique:", dim_customer["customer_key"].is_unique)
print("Dim_Product product_key unique:", dim_product["product_key"].is_unique)
print("Dim_Seller seller_key unique:", dim_seller["seller_key"].is_unique)
print("Dim_Date date_key unique:", dim_date["date_key"].is_unique)

print("\n===== FACT GRAIN VALIDATION =====")

fact_order_items_duplicates = fact_order_items[["order_id", "order_item_id"]].duplicated().sum()
fact_order_summary_duplicates = fact_order_summary["order_id"].duplicated().sum()

print("Fact_Order_Items duplicate grain rows:", fact_order_items_duplicates)
print("Fact_Order_Summary duplicate order_id rows:", fact_order_summary_duplicates)

print("\n===== NULL KEY VALIDATION =====")

print("Fact_Order_Items null customer_key:", fact_order_items["customer_key"].isnull().sum())
print("Fact_Order_Items null product_key:", fact_order_items["product_key"].isnull().sum())
print("Fact_Order_Items null seller_key:", fact_order_items["seller_key"].isnull().sum())
print("Fact_Order_Items null purchase_date_key:", fact_order_items["purchase_date_key"].isnull().sum())

print("Fact_Order_Summary null customer_key:", fact_order_summary["customer_key"].isnull().sum())
print("Fact_Order_Summary null purchase_date_key:", fact_order_summary["purchase_date_key"].isnull().sum())

print("\n===== REVENUE VALIDATION =====")

source_item_total = (order_items["price"] + order_items["freight_value"]).sum()
fact_item_total = fact_order_items["item_total_value"].sum()

print("Source item total:", round(source_item_total, 2))
print("Fact item total:", round(fact_item_total, 2))
print("Difference:", round(source_item_total - fact_item_total, 2))

===== DIMENSION KEY VALIDATION =====
Dim_Customer customer_key unique: True
Dim_Product product_key unique: True
Dim_Seller seller_key unique: True
Dim_Date date_key unique: True

===== FACT GRAIN VALIDATION =====
Fact_Order_Items duplicate grain rows: 0
Fact_Order_Summary duplicate order_id rows: 0

===== NULL KEY VALIDATION =====
Fact_Order_Items null customer_key: 0
Fact_Order_Items null product_key: 0
Fact_Order_Items null seller_key: 0
Fact_Order_Items null purchase_date_key: 0
Fact_Order_Summary null customer_key: 0
Fact_Order_Summary null purchase_date_key: 0

===== REVENUE VALIDATION =====
Source item total: 15843553.24
Fact item total: 15843553.24
Difference: 0.0


In [17]:
import os
import zipfile
from IPython.display import FileLink, display

# ============================================
# DOWNLOAD DATA MODEL TABLES DIRECTLY FROM JUPYTER
# ============================================

# 1. Create output folder in current Jupyter directory
output_folder = "olist_model_output"
os.makedirs(output_folder, exist_ok=True)

# 2. Store all final model DataFrames in dictionary
model_tables = {
    "dim_customer.csv": dim_customer,
    "dim_product.csv": dim_product,
    "dim_seller.csv": dim_seller,
    "dim_date.csv": dim_date,
    "fact_order_items.csv": fact_order_items,
    "fact_order_summary.csv": fact_order_summary
}

# 3. Save each DataFrame as CSV inside output folder
for file_name, df in model_tables.items():
    file_path = os.path.join(output_folder, file_name)
    df.to_csv(file_path, index=False)
    print(f"✅ Saved: {file_name}")

# 4. Create ZIP file
zip_file_name = "olist_data_model_tables.zip"

with zipfile.ZipFile(zip_file_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_name in model_tables.keys():
        file_path = os.path.join(output_folder, file_name)
        zipf.write(file_path, arcname=file_name)
        print(f"✅ Added to ZIP: {file_name}")

print("\n✅ ZIP file created successfully")

# 5. Display clickable download link
display(FileLink(zip_file_name))

✅ Saved: dim_customer.csv
✅ Saved: dim_product.csv
✅ Saved: dim_seller.csv
✅ Saved: dim_date.csv
✅ Saved: fact_order_items.csv
✅ Saved: fact_order_summary.csv
✅ Added to ZIP: dim_customer.csv
✅ Added to ZIP: dim_product.csv
✅ Added to ZIP: dim_seller.csv
✅ Added to ZIP: dim_date.csv
✅ Added to ZIP: fact_order_items.csv
✅ Added to ZIP: fact_order_summary.csv

✅ ZIP file created successfully


C:\Users\10848018\AppData\Local\Python\pythoncore-3.14-64\olist_data_model_tables.zip